In [1]:
# .env 파일에 저장된 API 키 등을 로드하여 환경 변수로 설정합니다.
from dotenv import load_dotenv
# OpenAI API와 통신하기 위한 클라이언트 객체를 생성합니다.
from openai import OpenAI
# 실습 강의에서 제공하는 평가 보조 함수들을 불러옵니다.
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

# 환경 변수를 적용합니다 (예: OPENAI_API_KEY).
load_dotenv()
# OpenAI API 호출을 담당할 클라이언트 인스턴스를 생성합니다.
openai_client = OpenAI()

In [2]:
# pandas 라이브러리를 통해 데이터를 다룹니다.
import pandas as pd
# 평가를 위해 사전에 정의된 CSV 파일 경로들을 불러옵니다.
from evaluation_paths import RAG_ANSWERS_CSV, RAG_EVALUATIONS_CSV

# 이전 단계(03-rag-evaluation.ipynb)에서 생성한 395개의 RAG 답변 데이터셋을 로드합니다.
df_answers = pd.read_csv(RAG_ANSWERS_CSV)

# 데이터프레임 형식을 반복문이나 LLM Judge 호출에 용이한 리스트-딕셔너리 구조로 변환합니다.
answers = df_answers.to_dict(orient="records")

In [3]:
from pydantic import BaseModel, Field # 데이터 검증 및 구조 정의를 위한 라이브러리
from typing import Literal # 특정 값만 허용하도록 타입을 제한하는 라이브러리

# LLM 평가자의 답변 결과를 담을 데이터 구조를 정의합니다.
class AnswerEvaluation(BaseModel):
    # Field: 데이터의 제약 조건과 설명을 정의합니다.
    # LLM이 이 설명을 읽고 어떤 내용을 작성해야 할지 이해하게 됩니다.
    reasoning: str = Field(
        description="답변의 품질에 대한 추론 및 근거를 작성합니다."
    )
    
    # Literal: 변수가 가질 수 있는 값을 제한합니다.
    # LLM은 'good' 또는 'bad' 중 하나만 출력해야 합니다.
    score: Literal["good", "bad"] = Field(
        description="답변이 정확하고 완벽하면 'good', 그렇지 않으면 'bad'를 선택합니다."
    )

In [4]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [5]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [6]:
# 전체 395개 데이터가 들어있는 answers 리스트에서 첫 번째(index 0) 데이터를 가져옵니다.
rec = answers[0]

# 가져온 데이터(질문, 원본 답변, RAG 답변)의 내용을 확인합니다.
rec

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [7]:
# AQA(Answer-Question-Answer) 방식의 평가를 수행하는 함수를 정의합니다.
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    # 전달받은 데이터(질문, 원본답변, AI답변)를 평가용 프롬프트 템플릿에 채워 넣습니다.
    prompt = aqa_judge_prompt.format(
    question=question,     # 사용자의 원본 질문을 프롬프트에 할당
    answer_orig=answer_orig, # 비교 대상인 정답 데이터를 프롬프트에 할당
    answer_llm=answer_llm    # 평가 대상인 LLM의 답변을 프롬프트에 할당
) # 템플릿의 각 항목에 데이터를 채워 최종 평가 문장(프롬프트) 완성

    # 정의된 구조(AnswerEvaluation)대로 결과를 반환받기 위해 LLM을 호출합니다.
    # 만약 호출에 실패하면 지정된 횟수만큼 자동으로 재시도합니다.
    result, usage = llm_structured_retry(
        openai_client,           # API 연결 클라이언트
        aqa_judge_instructions,  # LLM 평가자의 평가 지침(System Prompt)
        prompt,                  # 완성된 평가 프롬프트
        AnswerEvaluation,        # 출력 데이터의 스키마(형식)
        model=model,             # 사용할 모델명
    )

    # 평가 결과(객체)와 사용량(usage) 정보를 반환합니다.
    return result, usage

In [8]:
# aqa_judge_prompt 템플릿의 { } 위치에 rec 딕셔너리의 값들을 각각 채워 넣습니다.
prompt = aqa_judge_prompt.format(
    question=rec["question"],     # 사용자의 질문 내용을 넣습니다.
    answer_orig=rec["answer_orig"], # 비교 기준이 될 정답(ground truth)을 넣습니다.
    answer_llm=rec["answer_llm"]    # 평가 대상인 RAG 파이프라인의 생성 답변을 넣습니다.
)
# 완성된 프롬프트 내용을 화면에 출력하여, LLM에게 전달될 최종 형태를 검토합니다.
print(prompt)

Question:
Is it okay to join the course late if I just found it now?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.


In [9]:
# [기존 코드 주석 처리 - API 호출 방지]
# API 사용량 소진으로 인한 에러를 방지하기 위해 해당 평가 루프를 주석 처리합니다.
# eval_result, usage = llm_structured_retry(
#     openai_client,               # API 요청을 보낼 클라이언트입니다.
#     aqa_judge_instructions,      # 평가 기준이 담긴 지침입니다.
#     prompt,                      # 평가할 질문과 답변이 담긴 프롬프트입니다.
#     AnswerEvaluation,            # 결과값을 정해진 구조로 받기 위한 스키마입니다.
# )
# eval_result                      # 평가 결과 객체를 출력합니다.

# [대체 코드] 기존 평가 결과 CSV 로드 및 데이터 구조 복구
import pandas as pd                 # 데이터 분석을 위한 판다스 라이브러리를 불러옵니다.
from evaluation_paths import RAG_EVALUATIONS_CSV # 이미 저장된 평가 결과 파일 경로를 가져옵니다.

# CSV 파일에서 이전 실습 때 생성된 평가 데이터들을 불러옵니다.
df_eval = pd.read_csv(RAG_EVALUATIONS_CSV)

# 데이터가 제대로 로드되었는지 확인하기 위해 전체 개수를 출력합니다.
print(f"기존 평가 결과 {len(df_eval)}건을 성공적으로 로드했습니다.")
# 데이터프레임의 상위 5개 행을 출력하여 데이터의 구조를 시각적으로 확인합니다.
df_eval.head()

기존 평가 결과 395건을 성공적으로 로드했습니다.


,question,document,score,reasoning
0,Is it okay to join the course late if I just f...,74eb249bbf,good,The AI answer preserves the ground truth meani...
1,Can I still take this course even if I missed ...,74eb249bbf,good,The AI answer preserves the core meaning: you ...
2,If I join after the course has already started...,74eb249bbf,good,The AI answer preserves the key point: joining...
3,Do I need to submit my project before submissi...,74eb249bbf,good,The AI answer preserves the key point of the g...
4,I’m a bit late to the course—what do I need to...,74eb249bbf,good,The AI answer includes the core requirement fr...


In [10]:
# [참고] 이 코드는 llm_structured_retry로부터 반환된 usage 정보를 바탕으로
# 해당 호출에 든 비용을 계산하는 함수입니다.

# calc_price(usage)  # 현재 API 호출을 수행하지 않으므로 실행 시 오류가 발생할 수 있습니다.

In [11]:
# [기존 코드 주석 처리 - API 호출 방지]
# evaluate_aqa 함수 호출을 막아 불필요한 API 비용 발생 및 에러를 방지합니다.
# eval_result, usage = evaluate_aqa(
#      question=rec["question"],    # 평가할 질문 데이터를 전달
#      answer_orig=rec["answer_orig"], # 정답지 데이터를 전달
#      answer_llm=rec["answer_llm"]    # 모델의 생성 답변을 전달
#  )
# eval_result                      # API로부터 받은 평가 결과를 출력

# # [대체 코드] CSV에서 해당 질문의 평가 결과만 조회
# # 앞서 로드한 df_eval에서 현재 질문(rec["question"])과 일치하는 행을 필터링합니다.
match = df_eval[df_eval['question'] == rec['question']] # 질문 컬럼을 비교해 조건에 맞는 행만 추출

if not match.empty: # 검색된 행이 하나라도 존재하는지 확인합니다.
    # 조회된 행의 score와 reasoning을 가져와 eval_result 형태와 유사하게 구성합니다.
    from types import SimpleNamespace # 객체의 속성처럼 데이터에 접근하기 위한 클래스를 임포트합니다.
    eval_result = SimpleNamespace( # 기존 코드의 결과 객체 구조를 흉내 냅니다.
        score=match.iloc[0]['score'],  # 결과 데이터프레임의 첫 번째 행에서 점수 값을 가져옵니다.
        reasoning=match.iloc[0]['reasoning'] # 결과 데이터프레임의 첫 번째 행에서 평가 근거를 가져옵니다.
    )
    print("평가 결과를 성공적으로 로드했습니다.") # 로드 성공 메시지를 출력합니다.
    print(eval_result) # 복구된 eval_result 객체를 출력하여 확인합니다.
else:
    print("해당 질문에 대한 평가 결과를 찾을 수 없습니다.") # 일치하는 질문이 없을 경우 에러 메시지를 출력합니다.

평가 결과를 성공적으로 로드했습니다.
namespace(score='good', reasoning='The AI answer preserves the ground truth meaning: late enrollment is allowed, but certificate eligibility requires submitting the project before submissions close. It is semantically equivalent.')


In [12]:
# 레코드 하나를 입력받아 평가를 수행하고 결과를 반환하는 함수를 정의합니다.
def judge_record(rec):
    # evaluate_aqa 함수를 호출하여 LLM으로부터 평가 결과와 토큰 사용량을 가져옵니다.
    eval_result, usage = evaluate_aqa(
        question=rec["question"],     # 입력받은 레코드의 질문을 할당합니다.
        answer_orig=rec["answer_orig"], # 정답지(ground truth)를 할당합니다.
        answer_llm=rec["answer_llm"]    # AI가 생성한 답변을 할당합니다.
    )

    # 평가된 결과와 원본 데이터를 조합하여 새로운 딕셔너리 구조를 생성합니다.
    result = {
        "question": rec["question"],   # 원본 질문을 그대로 저장합니다.
        "document": rec["document"],   # 답변 생성에 사용된 문서를 저장합니다.
        "score": eval_result.score,    # LLM이 판정한 점수(good/bad)를 저장합니다.
        "reasoning": eval_result.reasoning, # LLM이 작성한 평가 근거를 저장합니다.
    }

    # 최종 결과물(딕셔너리)과 호출 시 발생한 토큰 사용량(usage)을 반환합니다.
    return result, usage

In [13]:
# [상단 주석: 멀티스레드 기반 평가 수행 루프]
# 강의에서 다루는 멀티스레드 평가 코드는 API 크레딧을 대량으로 소모하므로 실행을 방지합니다.
# [기존 코드 주석 처리 - API 호출 방지]
# from concurrent.futures import ThreadPoolExecutor # 비동기 처리를 위한 스레드 풀 실행기 임포트
# with ThreadPoolExecutor(max_workers=6) as pool:   # 6개의 스레드를 사용하여 병렬로 평가 작업 수행
#     results = map_progress(pool, answers, judge_record) # 전체 질문(answers)에 대해 judge_record 함수를 실행하고 진행바 표시

# [상단 주석: CSV 데이터를 활용한 평가 결과 대체 로드]
# API 호출을 건너뛰기 위해 이전에 저장된 평가 파일(CSV)에서 결과를 리스트 형태로 복원합니다.
import pandas as pd # 데이터 처리를 위해 pandas 라이브러리 사용
from evaluation_paths import RAG_EVALUATIONS_CSV # 평가 결과 CSV 파일 경로 정보 불러오기

# CSV 파일에서 저장된 평가 결과 데이터를 읽어옵니다.
df_eval_loaded = pd.read_csv(RAG_EVALUATIONS_CSV) # CSV를 데이터프레임으로 로드

# [상단 주석: 원본 데이터 구조(results) 복구]
# CSV 행을 순회하며 강의에서 요구하는 (결과딕셔너리, 사용량) 튜플 형태로 변환합니다.
results = [
    (row.to_dict(), None) # 각 행을 딕셔너리로 변환하고, 미확인 사용량은 None 처리
    for _, row in df_eval_loaded.iterrows() # 데이터프레임의 모든 행을 반복 순회
]

print(f"총 {len(results)}개의 평가 레코드를 CSV에서 성공적으로 로드했습니다.") # 로드된 레코드 총개수 출력

총 395개의 평가 레코드를 CSV에서 성공적으로 로드했습니다.


In [16]:
# 전체 평가 결과 리스트(results) 중 인덱스 10번(11번째) 데이터를 가져옵니다.
# 결과 리스트의 각 요소는 (평가 결과 딕셔너리, API 사용량)의 튜플 구조로 되어 있습니다.
results[10]

({'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
  'document': '489dd1c9d9',
  'score': 'good',
  'reasoning': "The AI answer matches the ground truth: it states students don't need the Zoom link, that participation is via YouTube Live, the URL is posted in the Telegram/Slack announcements channel, questions go through Slido, and the YouTube channel is available. It omits the caution about not posting questions in chat, but that is ancillary and doesn't change the core answer."},
 None)

In [17]:
# [상단 주석: 결과 데이터 분리 작업]
# 전체 리스트(results)에 묶여 있던 평가 결과와 API 사용량 정보를 각각의 전용 리스트로 분리합니다.

# 평가 결과(result_dict)를 담을 빈 리스트를 초기화합니다.
evaluations = [] 
# API 사용량(usage_dict)을 담을 빈 리스트를 초기화합니다.
usages = []      

# 전체 results 리스트를 순회하며 튜플 형태의 데이터를 각각의 리스트에 담습니다.
for evaluation, usage in results: # 각 튜플에서 평가 결과와 사용량을 언패킹하여 할당
    evaluations.append(evaluation) # 평가 딕셔너리만 evaluations 리스트에 추가
    usages.append(usage)           # 사용량 데이터만 usages 리스트에 추가

In [19]:
# [상단 주석: 총 API 비용 계산 함수]
# 사용된 모든 토큰의 비용을 합산하여 최종 API 요금을 계산하는 함수입니다.

# usages 리스트에 담긴 각 호출별 사용량 정보를 바탕으로 전체 비용을 산출합니다.
calc_total_price(usages) # 각 평가 단계에서 발생한 API 사용료(usage)의 총합을 계산하여 반환

0.0

In [20]:
# [상단 주석: 평가 결과 리스트를 데이터프레임으로 변환]
# 리스트에 담겨 있던 개별 평가 결과 딕셔너리들을 하나의 표(데이터프레임) 형태로 결합하여,
# 전체 데이터를 한눈에 확인하고 통계 분석을 수행하기 쉽게 만듭니다.

df_eval = pd.DataFrame(evaluations) # evaluations 리스트를 판다스 데이터프레임 객체로 생성

In [21]:
# [상단 주석: 데이터프레임 구조 확인]
# 분석을 위해 준비된 전체 평가 데이터프레임(df_eval)의 상위 5개 행을 출력하여 데이터의 구성을 파악합니다.

df_eval.head() # 데이터프레임의 첫 5개 행을 출력

,question,document,score,reasoning
0,Is it okay to join the course late if I just f...,74eb249bbf,good,The AI answer preserves the ground truth meani...
1,Can I still take this course even if I missed ...,74eb249bbf,good,The AI answer preserves the core meaning: you ...
2,If I join after the course has already started...,74eb249bbf,good,The AI answer preserves the key point: joining...
3,Do I need to submit my project before submissi...,74eb249bbf,good,The AI answer preserves the key point of the g...
4,I’m a bit late to the course—what do I need to...,74eb249bbf,good,The AI answer includes the core requirement fr...


In [23]:
# [상단 주석: 점수 분포 집계]
# 'score' 컬럼의 값(good/bad)별로 빈도수를 계산하여 평가 결과의 전반적인 품질을 파악합니다.

df_eval.score.value_counts() # score 컬럼의 각 값별 개수를 집계

score
good    379
bad      16
Name: count, dtype: int64

In [24]:
# [상단 주석: 점수 분포 비율 계산]
# 'score' 컬럼의 값(good/bad)별 빈도수를 전체 대비 비율(0~1 사이)로 계산하여 정규화된 분포를 확인합니다.

df_eval.score.value_counts(normalize=True) # normalize=True 옵션을 통해 단순 개수가 아닌 비율(%)을 출력합니다.

score
good    0.959494
bad     0.040506
Name: proportion, dtype: float64

In [25]:
# [상단 주석: 부정적 평가 항목 조회]
# score가 'bad'인 레코드만 필터링하여, 어떤 부분에서 답변이 불만족스러웠는지 분석합니다.

# 'score' 컬럼의 값이 'bad'인 행만 추출하여 데이터프레임을 필터링하고 상위 5개를 출력
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
15,How do the free GPU hours work on these cloud ...,c6c2888275,bad,The AI answer does not convey the ground truth...
29,Is peer-review of the capstone project require...,69d122f12e,bad,The AI answer is not semantically equivalent t...
38,How will I know when a module is actually read...,96286b4be4,bad,The AI answer does not convey the ground truth...
73,Which model should I use in chat.completions.c...,152af39a53,bad,The ground truth says the issue is insufficien...
106,Do I need an OpenAI API key just to check how ...,fe8fed31e6,bad,The AI answer does not address the question or...


In [57]:
# [상단 주석: 데이터프레임 CSV 파일로 저장]
# 분석이 완료된 데이터프레임(df_eval)을 추후 재사용할 수 있도록 CSV 파일로 저장합니다.

df_eval.to_csv(RAG_EVALUATIONS_CSV, index=False) # 데이터프레임을 지정된 경로(RAG_EVALUATIONS_CSV)에 인덱스 정보 제외하고 저장